# Tel Aviv Business Data Ingestion - Bronze Layer

## Overview
This notebook implements the **Bronze (Raw) layer** of data pipeline for Tel Aviv municipality business data. It ingests raw data from external sources and stores it in an immutable, append-only format with full audit trails.

## Purpose
* Ingest business registry data from Tel Aviv municipality ArcGIS API

## Key Features
* **Idempotent Ingestion**: Uses Delta merge with `event_id` deduplication - reruns don't create duplicates
* **Event-Based Deduplication**: Deterministic `event_id` generated via SHA-256 hash of payload
* **Audit Trail**: Captures ingestion timestamp, source system, and original payload
* **Error Handling**: Graceful handling of network errors and API failures

## Data Sources
* **Business Registry**: Tel Aviv ArcGIS REST API (MapServer endpoint)

## Output Tables
* `workspace.tel_aviv_municipality_raw.business` - Raw business registry data

## Design Principles
* **Immutable Raw Data**: Bronze tables preserve original payload as JSON strings
* **Change Data Feed Enabled**: Supports incremental downstream processing
* **Schema-on-Read**: Minimal transformation at ingestion - preserve all source data

In [0]:
import json
import requests
import uuid
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, IntegerType, DateType
from pyspark.sql import Row, functions as F
from delta.tables import DeltaTable

In [0]:
def ingest_business_bronze_payload():
    """
    Ingests municipal business data from the Tel Aviv ArcGIS REST API into the Bronze Delta table.
    
    This function implements an idempotent 'Schema-on-Read' pattern by capturing API 
    attributes as a raw JSON payload. This ensures that upstream schema changes do not 
    break the ingestion pipeline.

    Logic:
    1. Fetches real-time business data from the Tel Aviv GIS server.
    2. Encapsulates each feature's attributes into a JSON string.
    3. Generates a deterministic 'event_id' using SHA-256 for deduplication.
    4. Performs an idempotent MERGE: Inserts records only if the event_id is new, 
       ensuring no duplicate records are added if the API returns the same data.

    Metadata Added:
    - uuid: Unique identifier for the specific API call.
    - ingestion_at: Timestamp of when the data hit the lakehouse.
    - source_system: The origin API (ArcGIS_TelAviv).
    - event_id: Hash used for idempotency and downstream deduplication.

    Returns:
        str: A status message indicating the number of processed features or error details.
    """
    api_url = "https://gisn.tel-aviv.gov.il/arcgis/rest/services/IView2/MapServer/925/query?where=1%3D1&outFields=*&f=json"
    destination_table = "workspace.tel_aviv_municipality_raw.business"
    
    try:
        # Fetch data from API
        response = requests.get(api_url, timeout=(3.1, 30))
        response.raise_for_status() 
        
        data = response.json()
        features = data.get('features', [])
        assert len(features) > 0, "API returned 0 features."

        # Map to Row objects
        raw_data = [
            Row(
                uuid=str(uuid.uuid4()), 
                payload=json.dumps(f['attributes'])
            ) 
            for f in features
        ]

        # Create DataFrame & generate the deterministic event_id
        df_new = spark.createDataFrame(raw_data) \
            .withColumn("ingestion_at", F.current_timestamp()) \
            .withColumn("source_system", F.lit("ArcGIS_TelAviv")) \
            .withColumn("event_id", F.sha2(F.col("payload"), 256))

        # Perform Delta Merge
        if spark.catalog.tableExists(destination_table):
            target_table = DeltaTable.forName(spark, destination_table)
            
            (target_table.alias("target")
             .merge(
                 df_new.alias("updates"),
                 "target.event_id = updates.event_id"
             )
             # If matched, we do nothing
             .whenNotMatchedInsertAll()
             .execute())
            
            return f"Success: Processed {len(features)} records. Only new event_ids were inserted into {destination_table}."
        
        else:
            # For first time: Create table
            df_new.write.format("delta").mode("overwrite").saveAsTable(destination_table)
            return f"Success: Table created. Inserted {len(features)} records."

    except requests.exceptions.RequestException as e:
        return f"Error: Network error: {e}"
    except Exception as e:
        return f"Error: {e}"

In [0]:
# Execute
print(ingest_business_bronze_payload())